# ⚽ Real-Time Ball Detection - YOLOv8 Training & ONNX Export

This notebook allows you to upload your downloaded Roboflow `.zip` dataset, fine-tune **YOLOv8n** on GPU, and export `best.onnx` (320x320 resolution) for fast CPU inference.

### Step 1: Check GPU & Install Dependencies

In [ ]:
!nvidia-smi
%pip install ultralytics onnxruntime opencv-python numpy scikit-learn onnx onnxsim

### Step 2: Upload & Extract Roboflow Dataset `.zip` File
Run the cell below, click **Choose Files**, and select your downloaded dataset `.zip` file.

In [ ]:
import os
import zipfile
from google.colab import files

dataset_dir = "/content/dataset"
os.makedirs(dataset_dir, exist_ok=True)

# Check if a zip is already uploaded in /content or prompt upload
zip_files = [f for f in os.listdir("/content") if f.endswith(".zip")]

if not zip_files:
    print("📁 Please click 'Choose Files' below and select your downloaded dataset .zip file:")
    uploaded = files.upload()
    zip_files = list(uploaded.keys())

if zip_files:
    zip_path = os.path.join("/content", zip_files[0])
    print(f"📦 Extracting {zip_path} into /content/dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
    print("✅ Dataset extracted successfully to /content/dataset!")
else:
    print("⚠️ No zip file uploaded!")

### Step 3: Run Training & ONNX Export

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

data_yaml_path = "/content/dataset/data.yaml"

if not os.path.exists(data_yaml_path):
    # Search if data.yaml is inside a subfolder inside /content/dataset
    yaml_search = list(Path(dataset_dir).rglob("data.yaml"))
    if yaml_search:
        data_yaml_path = str(yaml_search[0])

if not os.path.exists(data_yaml_path):
    print(f"⚠️ Error: data.yaml not found inside /content/dataset! Please check dataset contents.")
else:
    print(f"📄 Using data configuration: {data_yaml_path}")
    # Load pretrained YOLOv8n
    model = YOLO("yolov8n.pt")

    # Fine-tune model
    results = model.train(
        data=data_yaml_path,
        epochs=50,
        imgsz=640,
        batch=16,
        optimizer="AdamW",
        lr0=0.01,
        lrf=0.01,
        cos_lr=True,
        patience=15,
        seed=42,
        device=0
    )

    # Get best weights
    best_pt = Path(results.save_dir) / "weights" / "best.pt"
    trained_model = YOLO(str(best_pt))

    # Export to ONNX at 320x320 for real-time CPU performance
    exported_path = Path(
        trained_model.export(
            format="onnx",
            imgsz=320,
            opset=17,
            simplify=True
        )
    )

    output_path = Path("/content/best.onnx")
    output_path.write_bytes(exported_path.read_bytes())
    print(f"✅ Training complete! Model exported to {output_path}")

### Step 4: Download best.onnx to your Computer

In [ ]:
from google.colab import files

# Downloads best.onnx straight to your computer's Downloads folder
files.download("/content/best.onnx")